# 06 — Full Pipeline Demo (Severity + Similarity)

This notebook runs **one combined prediction flow** and prints:
- Similar bridge cases
- Retrofit recommendation
- Severity / risk prediction

## Training (do this first)
Run these notebooks (they create timestamped checkpoints under `artifacts/`):
- **03 — Severity Model** (creates `severity_model.joblib` + `preprocessors.joblib`)
- **04 — Similarity Model** (creates `similarity_index.joblib` + `case_table.csv` + `preprocessors.joblib`)

No new folders are created by the demo below — it only **loads the latest saved checkpoints**.

## Demo / Inference
Provide a JSON object of feature values (partial is OK).
The pipeline automatically loads the newest severity checkpoint and newest similarity checkpoint from `artifacts/` (no copying).

In [9]:
from google.colab import drive

drive.mount('/content/drive')

%cd /content/drive/MyDrive/retrofit

PROJECT_ROOT = '/content/drive/MyDrive/retrofit'
CONFIG = 'configs/colab_large.yaml'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/retrofit


In [10]:
!pip -q install -r requirements-colab.txt
!pip -q install -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for bridge-retrofit (pyproject.toml) ... done


# Code to test the model

In [14]:
import json
import subprocess
import sys
import pandas as pd
from pathlib import Path

# Show which saved checkpoints will be used (timestamp folders)
artifacts_root = Path(PROJECT_ROOT) / "artifacts"
run_dirs = sorted([d for d in artifacts_root.iterdir() if d.is_dir()], reverse=True)
severity_run = next(
    (d for d in run_dirs if (d / "severity_model.joblib").exists() and (d / "preprocessors.joblib").exists()),
    None,
 )
similarity_run = next(
    (
        d
        for d in run_dirs
        if (d / "similarity_index.joblib").exists() and (d / "case_table.csv").exists() and (d / "preprocessors.joblib").exists()
    ),
    None,
 )
print("Latest severity run:", severity_run.name if severity_run else None)
print("Latest similarity run:", similarity_run.name if similarity_run else None)

# One combined payload → combined output (severity + similarity)
payload = {
    "_comment": "Missing values are allowed; extra keys are ignored.",
    "Bridge_Type": "Beam",
    "Material": "Steel",
    "Age_Years": 25,
    "Span_Length_m": 30,
    "Number_of_Spans": 3,
    "Failure_Type": "Scour",
}

out = subprocess.check_output([
    sys.executable, '-m', 'bridge_retrofit.cli',
    '--config', CONFIG,
    '--project-root', PROJECT_ROOT,
    'predict',
    '--json', json.dumps(payload),
], text=True)
print(out)

result = json.loads(out)

# Severity / risk output
print("Severity prediction:", result.get("severity_pred"))
if result.get("severity_proba") is not None:
    print("Severity probabilities:", result.get("severity_proba"))

# Similarity output
print("Recommended retrofit:", result.get("recommended_retrofit"))
similar_cases = result.get("similar_cases") or []
if similar_cases:
    display(pd.DataFrame(similar_cases).head(10))
else:
    print("No similar cases returned (run Similarity training first).")

Latest severity run: 20260511T195003Z
Latest similarity run: 20260511T195254Z
{
  "severity_pred": "Minor",
  "severity_proba": [
    0.36619564535751997,
    0.28018917097937135,
    0.3536151836631086
  ],
  "similar_cases": [
    {
      "Bridge_Name": "MetroSpan_95713",
      "Failure_Type": "Bearing Failure",
      "Suggested_Retrofit": "Bearing replacement",
      "distance": 10.285947799682617
    },
    {
      "Bridge_Name": "RiverLink_43392",
      "Failure_Type": "Bearing Failure",
      "Suggested_Retrofit": "Bearing replacement",
      "distance": 13.729239463806152
    },
    {
      "Bridge_Name": "SkyBridge_52310",
      "Failure_Type": "Bearing Failure",
      "Suggested_Retrofit": "Bearing replacement",
      "distance": 17.234312057495117
    },
    {
      "Bridge_Name": "DeltaCross_53752",
      "Failure_Type": "Bearing Failure",
      "Suggested_Retrofit": "Bearing replacement",
      "distance": 17.706348419189453
    },
    {
      "Bridge_Name": "HeritageWay_17

,Bridge_Name,Failure_Type,Suggested_Retrofit,distance
0,MetroSpan_95713,Bearing Failure,Bearing replacement,10.285948
1,RiverLink_43392,Bearing Failure,Bearing replacement,13.729239
2,SkyBridge_52310,Bearing Failure,Bearing replacement,17.234312
3,DeltaCross_53752,Bearing Failure,Bearing replacement,17.706348
4,HeritageWay_17361,Bearing Failure,Bearing replacement,18.928246


In [ ]:
# Optional: launch the Gradio UI
# Note: this cell BLOCKS while the server is running (that is expected).
# Tip: `--share` can take a while in Colab because it creates a public tunnel URL.
import subprocess
import sys

subprocess.check_call([
    sys.executable, '-m', 'bridge_retrofit.cli',
    '--config', CONFIG,
    '--project-root', PROJECT_ROOT,
    'serve',
    '--share',
])

KeyboardInterrupt: 